# Which token is actually the surname?

Analysis 03 rests on a list I wrote by hand: names like *devi* and *kumari* that
sit in the surname slot without naming a family. A hand list is the weakest thing
in this repo, and the electoral rolls can measure it instead.

Every elector record carries a second name — the father's or the husband's. **A
token appearing in both names is one that passed between two family members,
which is what a family name is.** An honorific does not pass. Nor does a filler,
a patronymic, or an initial.

This notebook tests that idea, and it goes wrong twice before it goes right.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent / "03_how_few_names"))

import transmission as tr
import titles

CACHE = Path("out/tab")
pd.set_option("display.width", 120)

## 1. Is the field even there?

The idea needs the relative's name to be populated. It mostly is — and Delhi is
the exception that has none at all, which is why an earlier probe of Delhi found
almost nothing.

In [2]:
import csv, gzip

def coverage(state, rows=400_000):
    path = tr.roll_path(state)
    if not path.exists():
        return None
    total = present = 0
    with gzip.open(path, "rt") as fh:
        for i, row in enumerate(csv.DictReader(fh)):
            if i >= rows:
                break
            n = int(row["n_times"])
            total += n
            if (row.get("father_husband_name") or "").strip():
                present += n
    return present / max(total, 1)

pd.DataFrame(
    [{"state": s, "relative_name_present": coverage(s)}
     for s in ["bihar", "uttar_pradesh", "west_bengal", "kerala",
               "tamil_nadu", "maharashtra", "punjab", "delhi"]]
).set_index("state").style.format("{:.1%}")

,relative_name_present
state,
bihar,97.7%
uttar_pradesh,96.5%
west_bengal,91.6%
kerala,99.8%
tamil_nadu,96.2%
maharashtra,99.8%
punjab,99.7%
delhi,0.0%


## 2. Bihar: the split is clean, and Kumar is not on either side

Score each token by the share of its bearers for whom it also appears in the
relative's name.

In [3]:
bihar = tr.scan("bihar", limit=900_000)
tokens = tr.by_token(bihar, min_bearers=20_000)
tokens.head(16).style.format({"transmitted": "{:.1%}", "bearers": "{:,.0f}"}).hide(axis="index")

state,token,bearers,transmitted,modal_position
bihar,devi,"6,311,413",0.0%,last
bihar,kumar,"1,836,714",9.6%,last
bihar,kumari,"763,651",0.0%,last
bihar,singh,"602,201",99.7%,last
bihar,yadav,"413,874",99.9%,last
bihar,sah,"283,938",100.0%,last
bihar,sunita,"272,524",0.0%,first
bihar,khatun,"269,604",0.0%,none
bihar,ram,"268,981",89.4%,last
bihar,paswan,"238,427",100.0%,last


Family names sit at ~100%: singh, yadav, sah, paswan, ray, mahato, mandal,
thakur, sharma. Sex-marking names sit at 0: devi, kumari, khatun. Given names
that happen to be common, like *sunita*, sit at 0 too — correctly, since a
daughter does not inherit her father's given name.

**Kumar lands at about 11%**, and that is the useful part. Analysis 03 had to
decide whether Kumar was a family name or a title, and it is neither: for most
people it is the filler in the middle of `Abhay Kumar Sharma`, and for a real
minority it is the surname they actually carry. The measurement says so without
being asked.

In [4]:
bands = tokens.assign(
    band=pd.cut(tokens.transmitted, [0, .02, .3, .8, 1.01], right=False,
                labels=["never (<2%)", "rarely", "often", "almost always (>80%)"])
)
bands.groupby("band", observed=True).agg(
    tokens=("token", "size"), people=("bearers", "sum"),
    examples=("token", lambda s: ", ".join(s.head(6))),
)

,tokens,people,examples
band,,,
never (<2%),138,15998074,"devi, kumari, sunita, khatun, anita, geeta"
rarely,2,1861204,"kumar, alam"
almost always (>80%),20,2945360,"singh, yadav, sah, ram, paswan, ray"


## 3. Where the method can run at all

Before reading any score, ask whether the relative's name even contains a
surname. Sometimes it is a bare given name, and then there is nothing to match.

In [5]:
def relative_shape(state, rows=300_000):
    path = tr.roll_path(state)
    counts = {0: 0, 1: 0, 2: 0}
    total = 0
    with gzip.open(path, "rt") as fh:
        for i, row in enumerate(csv.DictReader(fh)):
            if i >= rows:
                break
            n = int(row["n_times"])
            total += n
            counts[min(len((row.get("father_husband_name") or "").split()), 2)] += n
    return {"state": state, "relative_absent": counts[0] / total,
            "relative_one_token": counts[1] / total,
            "relative_has_surname": counts[2] / total}

pd.DataFrame([relative_shape(s) for s in
              ["bihar", "maharashtra", "odisha", "west_bengal", "punjab",
               "uttar_pradesh", "rajasthan", "kerala", "tamil_nadu", "gujarat"]]
).set_index("state").style.format("{:.0%}")

,relative_absent,relative_one_token,relative_has_surname
state,,,
bihar,3%,0%,97%
maharashtra,0%,2%,98%
odisha,1%,1%,98%
west_bengal,9%,1%,90%
punjab,0%,1%,99%
uttar_pradesh,4%,66%,30%
rajasthan,1%,78%,22%
kerala,0%,93%,7%
tamil_nadu,4%,96%,1%


This is the correction that mattered most, and I got it wrong first.

In **Gujarat** the relative is recorded as a bare given name in *every* row —
`patel ramilaben` with father `rameshbhai`. The elector's surname is right there
in front, but there is nothing on the other side to match it to. Tamil Nadu is
96% the same way, Kerala 93%.

I had read Tamil Nadu's near-zero score as evidence that Tamil naming is
patronymic and leaves no family name to inherit. That was reading a recording
convention as a fact about India. The score is uninterpretable there, full stop.

**The method only runs where the relative's name carries a surname**: Bihar,
Maharashtra, Odisha, West Bengal and Punjab. Everything below is restricted to
those, and `method_applies` marks it in the table.

Within them, Punjab is still worth a note: singh scores ~97% and kaur ~0%. Singh
does descend father to son, so a title given to every Sikh man is
indistinguishable from a family name by this test. That is a limit to state, not
a bug to patch.

## 4. Where the obvious version breaks

My first scorer looked only at the **last** token, because that is what
`instate`'s `last_name` column holds and what analysis 03 inherited. On
Maharashtra it reported that the commonest surnames were *shankar*, *ashok*,
*lakshman*, *suresh* — all given names — each "shared with the father" about 96%
of the time.

Three rounds of aggregate statistics did not explain it. Reading eight raw rows
did.

In [6]:
with gzip.open(tr.roll_path("maharashtra"), "rt") as fh:
    rows = [r for i, r in zip(range(400_000), csv.DictReader(fh))
            if r["english_name"].endswith("ashok")][:6]
pd.DataFrame(rows)[["english_name", "father_husband_name", "n_times"]]

,english_name,father_husband_name,n_times
0,patil anita ashok,patil ashok,368
1,patil sunita ashok,patil ashok,342
2,patil manisha ashok,patil ashok,300
3,patil sandip ashok,patil ashok,287
4,patil amol ashok,patil ashok,283
5,patil sunanda ashok,patil ashok,283


**Maharashtra writes the surname first.** `patil ashwini`, father `patil ashok`.
The last token is a given name; the family name is in front. A last-token scorer
reads the wrong word for an entire state, and every aggregate built on it —
including analysis 03's "449 names cover half of Maharashtra" — is about given
names.

Searching every position fixes it, and the position itself becomes a finding.

In [7]:
by_tok, by_st = tr.scan_all(cache=CACHE)
by_st.assign(
    convention=lambda d: d[["position_first", "position_middle", "position_last"]]
    .idxmax(axis=1).str.replace("position_", "")
).sort_values("position_first", ascending=False)[
    ["state", "usable_share", "shared_share",
     "position_first", "position_last", "convention"]
].style.format({c: "{:.1%}" for c in
    ["usable_share", "shared_share", "position_first", "position_last"]}).hide(axis="index")

state,usable_share,shared_share,position_first,position_last,convention
maharashtra,97.5%,94.1%,96.6%,3.1%,first
gujarat,99.7%,0.0%,44.5%,48.9%,last
tamil_nadu,7.2%,6.4%,32.5%,57.1%,last
uttar_pradesh,44.0%,20.5%,9.9%,86.2%,last
bihar,97.1%,38.4%,6.8%,91.2%,last
kerala,38.0%,18.9%,5.3%,87.4%,last
rajasthan,32.9%,24.1%,4.0%,93.6%,last
west_bengal,92.2%,73.6%,2.5%,94.9%,last
odisha,98.4%,81.3%,1.3%,96.8%,last
punjab,89.8%,42.6%,1.2%,97.4%,last


## 5. The frequency floor, tested and rejected

A natural second filter: a token carried by very few people cannot be a surname.
It is a reasonable prior, and the data argues against it.

In [8]:
d = tr.by_token(bihar, min_bearers=1)
rare_but_transmitted = d[(d.bearers < 50) & (d.transmitted > 0.9)]
print("rare tokens that still transmit:", ", ".join(rare_but_transmitted.token.head(12)))

rare tokens that still transmit: vind, murmoo, soren, miya, hembram, marandi, bhuiyan, kevat, prajapat, modi, jamadar, chaurasiya


Those are real surnames. `soren`, `hembram`, `marandi`, `murmoo` are Santhal
names; `kevat`, `chaurasiya`, `bhuiyan`, `prajapat` are real castes. A floor
would delete exactly the tribal and regional names this repo should be most
careful with.

And the thing a floor was meant to catch does not happen. The worry is
correlated OCR — the same scanning error hitting an elector and their father on
one page, so a garbled token looks inherited.

In [9]:
from rapidfuzz.distance import Levenshtein
from rapidfuzz.process import cdist

head = d.nlargest(150, "bearers")
rare = d[d.bearers < 200]
dist = cdist(rare.token.tolist(), head.token.tolist(),
             scorer=Levenshtein.distance, score_cutoff=1, workers=-1)
near = dist.min(axis=1) <= 1
suspects = rare[near].assign(nearest=head.token.to_numpy()[dist[near].argmin(axis=1)])
print(f"{len(suspects)} rare tokens are one edit from a common one; "
      f"{(suspects.transmitted > 0.9).sum()} of them transmit")
suspects.nlargest(8, "bearers")[["token", "bearers", "transmitted", "nearest"]]

149 rare tokens are one edit from a common one; 4 of them transmit


,token,bearers,transmitted,nearest
1842,savila,198,0.0,savita
1402,tarun,192,0.0,arun
1674,sanjan,191,0.0,sanjay
1225,bikash,186,0.0,vikash
1595,laliya,184,0.0,lalita
1561,barti,183,0.0,aarti
1682,somni,180,0.0,soni
1385,samta,175,0.0,mamta


Almost all sit at zero, because they are garbled *given* names — `savila`,
`sanjan`, `bikash`, `munita` — and a given name does not transmit whatever its
spelling. So frequency belongs as a **reliability floor on the estimate**, not
in the definition: a token with three bearers can only score 0, ⅓, ⅔ or 1.

Normalisation still comes first. Analysis 03's `variants.py` folds `kamar` into
`kumar` before any of this.

## 6. The measurement against my hand list

The point of all this is to replace a judgment with a number. Here is where they
agree and where they do not.

In [10]:
hand = set(titles.CLEAR) | set(titles.AMBIGUOUS)
usable_states = set(by_st.loc[by_st.method_applies, "state"])
scored = by_tok[by_tok.state.isin(usable_states)]
print("states the method can run in:", ", ".join(sorted(usable_states)))

national = (scored.groupby("token")
            .apply(lambda g: pd.Series({
                "bearers": g.bearers.sum(),
                "transmitted": (g.transmitted * g.bearers).sum() / g.bearers.sum()}),
                include_groups=False)
            .sort_values("bearers", ascending=False))
national["on_hand_list"] = national.index.isin(hand)
national[national.transmitted < 0.25].head(25)[["bearers", "transmitted", "on_hand_list"]]

states the method can run in: bihar, maharashtra, odisha, punjab, rajasthan, uttar_pradesh, west_bengal


,bearers,transmitted,on_hand_list
token,,,
devi,35635066.0,0.001662,True
kumar,25994437.0,0.081458,True
kumari,5060081.0,0.001718,True
khatun,4931067.0,0.004304,True
kaur,4709507.0,0.003603,True
bibi,3503120.0,0.003907,True
lal,2513843.0,0.186582,False
begam,2041089.0,0.008325,True
sanjay,2022579.0,0.248053,False


In [11]:
missed = national[(national.transmitted < 0.25) & (~national.on_hand_list)].head(20)
wrong = national[(national.transmitted > 0.75) & (national.on_hand_list)]
print("On the list but measurably transmitted (my list was wrong):")
print(wrong[["bearers", "transmitted"]].to_string(), "\n")
print("Not on the list but measurably not transmitted (my list missed these):")
missed[["bearers", "transmitted"]]

On the list but measurably transmitted (my list was wrong):
          bearers  transmitted
token                         
singh  17413726.0     0.784854 

Not on the list but measurably not transmitted (my list missed these):


,bearers,transmitted
token,,
lal,2513843.0,0.186582
sanjay,2022579.0,0.248053
santosh,1870451.0,0.187048
sunita,1733408.0,0.002217
vijay,1650924.0,0.244275
chandra,1603003.0,0.100716
sunil,1571542.0,0.208429
anil,1433489.0,0.210141
anita,1298939.0,0.001818


## 7. What this cannot see

- **It only runs in five of the ten states scanned.** Everywhere else the roll
  records the relative as a bare given name, so there is no surname to match.
  A low score in Gujarat, Tamil Nadu or Kerala says nothing about naming there.
- **It measures patrilineal transmission.** A title given to every man of a
  community descends father to son and scores as a family name. Punjab's *singh*
  is the clear case.
- **Married women are matched against a husband**, so the score mixes
  inheritance with marriage. A woman who took her husband's family name counts
  as transmitting it.
- **Delhi cannot be scored at all** — the relative's name is empty in every row.
- **OCR depresses every share slightly**, since a misread on either side breaks
  a match that really existed.
- **A token can be a family name in one place and not another**, which is why the
  tables are per state and the roll-up above is restricted.